# Waiting vs. Computing: Picking Threads, Asyncio or Processes

### Why can one server juggle 5,000 waiting tasks, yet choke on 150 CPU-heavy ones?

The answer depends on what a task *does* with its time. This notebook tests three claims:

1. **Waiting work.** Threads and asyncio both overlap waiting. A capped thread pool must handle extra work in waves; an event loop can hold thousands of waiting tasks at once.
2. **CPU work in pure Python.** Threads and asyncio give no useful parallelism on a standard GIL-enabled CPython.
3. **Process workers.** Separate processes can use several cores, but the real speedup depends on how many cores you can actually use and how big each job is.

Only the standard library is used. Timings differ per machine, so focus on the *pattern* in the numbers, not the exact ratios.

## Change map (reviewer's guide)

This notebook is a reorganised rewrite of `Copy_of_episode_2_colab.ipynb`. The table below shows where it differs from the original.

| This notebook | Original | What changed |
|---|---|---|
| Title + intro | Cell 0 | Reworded; same three claims. |
| Change map | – | This table. |
| Expected results | Cell 1 | Hypotheses turned into a "prediction" table. |
| Environment check | Cell 2 | Same checks, wrapped in `inspect_environment()` with an `import` cleanup. |
| Settings | Cells 3–4 | Merged into one section; same values, comments rewritten. |
| Helpers | Cells 5–6 | `timed` / `timed_async` share one `_summarise()`; prime code is stored once as `PRIME_SOURCE` (used locally *and* written to the worker file). |
| `await` pitfall | Cells 7–8 | Reworded; code logic identical. |
| Exp. 1A, 1B | Cells 9–12 | Same experiments; warm-up and shutdown moved into one `run_thread_vs_async()` helper. |
| Exp. 2 | Cells 13–14 | Reworded text; same three CPU runners. |
| Exp. 3 | Cells 15–16 | Worker file now built from `PRIME_SOURCE` (no duplicated function). |
| Correct `await` pattern | Cell 17 | Now sits directly after Experiment 3, next to the pool code it explains. |
| Speedup table | – | Computes each result relative to its baseline. |
| Summary + conclusions | Cells 18–20 | Reworded; caveats regrouped. |
| Metadata | `episode`, `q002`, `colab` keys | Removed (bookkeeping from the original runner). Kernelspec kept. |
| Outputs | Saved outputs from the original run | Replaced with outputs from a fresh full-size run on a 1-CPU Linux sandbox (Python 3.12). Re-run on your own machine for your own numbers; the process pool only shows a speedup when more than one core is usable. |

## What we expect to see

| Experiment | Prediction |
|---|---|
| 1A: equal concurrency (256 waits) | Threads and asyncio both land near the 50 ms floor; any gap is just runtime overhead. |
| 1B: 5,000 waits, 256 thread workers | Thread pool needs about 20 waves (~1 s). Asyncio keeps all 5,000 timers pending (~0.05 s plus overhead). |
| 2: CPU work, threads / asyncio | About the same as serial, because the GIL allows one thread to run Python at a time. |
| 3: CPU work, process pool | Faster than serial **if** enough cores are usable and each job is big enough to repay process overhead. |

## Step 0: Check the environment

In [1]:
# same checks as the original, grouped into one function.
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from pathlib import Path
from statistics import median
from time import perf_counter, sleep
import asyncio
import importlib
import math
import multiprocessing
import os
import platform
import sys
import tempfile


def inspect_environment():
    print(f"Python: {sys.version.split()[0]}")
    print(f"Platform: {platform.platform()}")
    print(f"Logical CPUs visible to Python: {os.cpu_count()}")

    probe = getattr(sys, "_is_gil_enabled", None)
    if probe:
        gil = probe()
    elif platform.python_implementation() == "CPython" and sys.version_info < (3, 13):
        gil = True
    else:
        gil = "unknown on this Python version/build"
    print(f"GIL enabled: {gil}")

    if hasattr(os, "sched_getaffinity"):
        allowed = os.sched_getaffinity(0)
        print(f"CPUs allowed by affinity: {len(allowed)} {allowed}")

    cpu_max = Path("/sys/fs/cgroup/cpu.max")
    if cpu_max.exists():
        quota, period = cpu_max.read_text(encoding="utf-8").strip().split()
        print(f"cgroup cpu.max: {quota} {period}")
        if quota == "max":
            print("No explicit cgroup CPU quota")
        else:
            print(f"Effective cgroup CPU quota: {int(quota) / int(period):.2f}")


inspect_environment()

Python: 3.12.3
Platform: Linux-6.18.44-fc-v139-x86_64-with-glibc2.39
Logical CPUs visible to Python: 1
GIL enabled: True
CPUs allowed by affinity: 1 {0}


## Step 1: Settings

The thread pool is capped at **256 workers**; it never creates 5,000 native threads. So the 5,000-task test compares *capacity*, not scheduler speed at equal load. That is why Experiment 1A exists.

In [2]:
# original cells 3-4 merged; values unchanged.

# Waiting workload
WAIT_TASKS = 5_000
WAIT_SECONDS = 0.050
THREAD_WAIT_WORKERS = 256
FAIR_WAIT_TASKS = THREAD_WAIT_WORKERS      # equal-concurrency probe

# CPU workload
CPU_TASKS = 150
PRIME_LIMIT = 16_000
CPU_THREAD_WORKERS = min(32, (os.cpu_count() or 1) + 4)

# Process workers: use every visible CPU, capped at four.
VISIBLE_CPUS = os.cpu_count() or 1
PROCESS_WORKERS = max(1, min(4, VISIBLE_CPUS))
# Real services usually start lower to leave headroom for the OS and network.
PRODUCTION_WORKER_START = max(1, min(4, VISIBLE_CPUS - 1))

REPEATS = 5

print({
    "waiting_tasks": WAIT_TASKS,
    "wait_seconds": WAIT_SECONDS,
    "bounded_thread_workers": THREAD_WAIT_WORKERS,
    "equal_concurrency_probe_tasks": FAIR_WAIT_TASKS,
    "cpu_tasks": CPU_TASKS,
    "prime_limit": PRIME_LIMIT,
    "cpu_thread_workers": CPU_THREAD_WORKERS,
    "experiment_process_workers": PROCESS_WORKERS,
    "production_worker_starting_point": PRODUCTION_WORKER_START,
    "repeats": REPEATS,
})

{'waiting_tasks': 5000, 'wait_seconds': 0.05, 'bounded_thread_workers': 256, 'equal_concurrency_probe_tasks': 256, 'cpu_tasks': 150, 'prime_limit': 16000, 'cpu_thread_workers': 5, 'experiment_process_workers': 1, 'production_worker_starting_point': 1, 'repeats': 5}


## Step 2: Measuring helpers

Each benchmark gets one untimed warm-up where practical, then five timed runs; we report the **median**. A gap under 5% is called "effectively equal" rather than treated as a real difference.

In [3]:
# timed/timed_async share _summarise(); prime code lives in PRIME_SOURCE.

# The prime counter is kept as source text so the SAME code runs in this
# notebook and in the worker file that process pools import (Experiment 3).
PRIME_SOURCE = """def count_primes(limit):
    count = 0
    for candidate in range(2, limit):
        prime = True
        divisor = 2
        while divisor * divisor <= candidate:
            if candidate % divisor == 0:
                prime = False
                break
            divisor += 1
        count += int(prime)
    return count
"""
exec(PRIME_SOURCE)   # defines count_primes() here


def _summarise(label, samples, last_value):
    return {
        "label": label,
        "seconds_median": median(samples),
        "seconds_all": samples,
        "value": last_value,
    }


def timed(label, fn, repeats=REPEATS):
    samples, value = [], None
    for _ in range(repeats):
        start = perf_counter()
        value = fn()
        samples.append(perf_counter() - start)
    return _summarise(label, samples, value)


async def timed_async(label, factory, repeats=REPEATS):
    samples, value = [], None
    for _ in range(repeats):
        start = perf_counter()
        value = await factory()
        samples.append(perf_counter() - start)
    return _summarise(label, samples, value)


def print_results(title, records):
    print(f"\n{title}")
    print("-" * len(title))
    width = max(len(r["label"]) for r in records)
    for r in records:
        samples = ", ".join(f"{x:.4f}" for x in r["seconds_all"])
        print(f"{r['label']:<{width}}  median={r['seconds_median']:.4f}s  samples=[{samples}]")


def describe_change(baseline, candidate):
    ratio = baseline["seconds_median"] / candidate["seconds_median"]
    if abs(ratio - 1.0) < 0.05:
        return f"{candidate['label']}: effectively equal to baseline ({ratio:.2f}x)"
    if ratio > 1:
        return f"{candidate['label']}: {ratio:.2f}x faster than baseline"
    return f"{candidate['label']}: {1 / ratio:.2f}x slower than baseline"


benchmark_results = {}

## A common mistake: `await count_primes(...)`

`await` needs an *awaitable*. But `count_primes()` does all the work immediately and returns a plain integer. By the time Python tries to await it, the calculation is already finished, and an `int` can't be awaited.

In [4]:
async def demonstrate_invalid_cpu_await():
    try:
        return await count_primes(2_000)
    except TypeError as error:
        return f"await failed as expected: {error}"


print(await demonstrate_invalid_cpu_await())

await failed as expected: object int can't be used in 'await' expression


## Experiment 1: Waiting work

`sleep()` stands in for "waiting". It isolates how each model schedules waits; it does **not** model sockets, payload sizes, database pools or backpressure.

**1A, equal concurrency.** Both models handle 256 simultaneous 50 ms waits. This is the fair comparison of overhead.

**1B, capacity.** Now they get different limits on purpose: the capped pool needs ~20 waves for 5,000 jobs, while the event loop parks all 5,000 at once. This shows *capacity*, not that asyncio is "N times faster" in general.

In [5]:
# warm-up, timing and shutdown folded into run_thread_vs_async().

def threaded_wait(pool, task_count):
    list(pool.map(lambda _: sleep(WAIT_SECONDS), range(task_count)))
    return task_count


async def async_wait(task_count):
    await asyncio.gather(*(asyncio.sleep(WAIT_SECONDS) for _ in range(task_count)))
    return task_count


async def run_thread_vs_async(task_count, thread_label, async_label):
    pool = ThreadPoolExecutor(max_workers=THREAD_WAIT_WORKERS)
    try:
        # Untimed warm-up: start every native worker, prime the event loop.
        list(pool.map(lambda _: sleep(0), range(THREAD_WAIT_WORKERS)))
        await async_wait(8)

        thread_rec = timed(thread_label, lambda: threaded_wait(pool, task_count))
        async_rec = await timed_async(async_label, lambda: async_wait(task_count))
    finally:
        pool.shutdown(wait=True)
    return thread_rec, async_rec


# --- 1A: equal concurrency ---
fair_thread, fair_async = await run_thread_vs_async(
    FAIR_WAIT_TASKS,
    f"threads ({FAIR_WAIT_TASKS} tasks / {THREAD_WAIT_WORKERS} workers)",
    f"asyncio ({FAIR_WAIT_TASKS} tasks / one loop thread)",
)
benchmark_results["waiting_equal"] = [fair_thread, fair_async]
print_results("1A: equal-concurrency waiting probe", benchmark_results["waiting_equal"])
print(describe_change(fair_thread, fair_async))


1A: equal-concurrency waiting probe
-----------------------------------
threads (256 tasks / 256 workers)      median=0.0619s  samples=[0.0822, 0.0619, 0.0619, 0.0587, 0.0680]
asyncio (256 tasks / one loop thread)  median=0.0545s  samples=[0.0559, 0.0545, 0.0536, 0.0536, 0.0569]
asyncio (256 tasks / one loop thread): 1.13x faster than baseline


In [6]:
# --- 1B: 5,000-task capacity comparison ---
thread_waves = math.ceil(WAIT_TASKS / THREAD_WAIT_WORKERS)
print(f"Thread-pool waves needed: {thread_waves}")
print(f"Thread wait floor:  ~{thread_waves * WAIT_SECONDS:.2f}s")
print(f"Asyncio wait floor: ~{WAIT_SECONDS:.2f}s plus task overhead")

scale_thread, scale_async = await run_thread_vs_async(
    WAIT_TASKS,
    f"bounded thread pool ({WAIT_TASKS} jobs / {THREAD_WAIT_WORKERS} workers)",
    f"asyncio ({WAIT_TASKS} outstanding tasks / one loop thread)",
)
benchmark_results["waiting_scale"] = [scale_thread, scale_async]
print_results("1B: 5,000-task waiting-capacity comparison", benchmark_results["waiting_scale"])
print("Note: this ratio includes the thread pool's deliberate 256-worker cap.")

Thread-pool waves needed: 20
Thread wait floor:  ~1.00s
Asyncio wait floor: ~0.05s plus task overhead

1B: 5,000-task waiting-capacity comparison
------------------------------------------
bounded thread pool (5000 jobs / 256 workers)       median=1.0602s  samples=[1.0309, 1.1273, 1.0495, 1.0841, 1.0602]
asyncio (5000 outstanding tasks / one loop thread)  median=0.1348s  samples=[0.1348, 0.1133, 0.1423, 0.1129, 0.1745]
Note: this ratio includes the thread pool's deliberate 256-worker cap.


## Experiment 2: CPU work in pure Python

Every task counts primes. No I/O, and no point where a task hands control back.

- **Serial** is the honest baseline.
- **Threads** share one GIL, so only one runs Python code at a time.
- **Asyncio `gather()`** wraps each call in a task, but a task with no `await` runs start to finish and blocks the loop. No parallelism appears.

This measures *throughput*. How late timers and socket callbacks become (event-loop lag) would need a separate experiment.

In [7]:
def serial_cpu_once():
    return [count_primes(PRIME_LIMIT) for _ in range(CPU_TASKS)]


def threaded_cpu_once():
    with ThreadPoolExecutor(max_workers=CPU_THREAD_WORKERS) as pool:
        return list(pool.map(count_primes, [PRIME_LIMIT] * CPU_TASKS))


async def async_cpu_once():
    async def one_task():
        # No await on purpose: this blocks the event-loop thread until done.
        return count_primes(PRIME_LIMIT)

    return await asyncio.gather(*(one_task() for _ in range(CPU_TASKS)))


# Untimed warm-ups
count_primes(PRIME_LIMIT)
with ThreadPoolExecutor(max_workers=2) as warm_pool:
    list(warm_pool.map(count_primes, [2_000, 2_000]))
await async_cpu_once()

serial_record = timed("serial", serial_cpu_once)
thread_record = timed(f"threads ({CPU_THREAD_WORKERS} workers)", threaded_cpu_once)
async_record = await timed_async("asyncio gather (CPU tasks do not yield)", async_cpu_once)

cpu_records = [serial_record, thread_record, async_record]
benchmark_results["cpu"] = cpu_records

# All three approaches must produce identical answers.
assert len({tuple(r["value"]) for r in cpu_records}) == 1

print_results("Pure-Python CPU workload", cpu_records)
print(describe_change(serial_record, thread_record))
print(describe_change(serial_record, async_record))


Pure-Python CPU workload
------------------------
serial                                   median=3.1641s  samples=[2.9173, 3.1988, 3.2124, 3.1641, 3.1636]
threads (5 workers)                      median=3.1305s  samples=[2.8499, 3.1305, 3.2184, 3.1402, 3.0902]
asyncio gather (CPU tasks do not yield)  median=3.0448s  samples=[2.9948, 3.2416, 3.9237, 3.0351, 3.0448]
threads (5 workers): effectively equal to baseline (1.01x)
asyncio gather (CPU tasks do not yield): effectively equal to baseline (1.04x)


## Experiment 3: A reusable pool of worker processes

Each process has its own interpreter and its own GIL, so pure-Python work can run on separate cores at the same time.

The pool is created once, warmed up and reused, so process start-up cost is not counted in every trial. We first time the pool directly (pure CPU throughput), then show the correct way to `await` it from the event loop.

In [8]:
# the worker module is written from PRIME_SOURCE (no copy-pasted function).
worker_dir = Path(tempfile.gettempdir()) / "workload_workers"
worker_dir.mkdir(parents=True, exist_ok=True)
(worker_dir / "workload_worker.py").write_text(PRIME_SOURCE, encoding="utf-8")

if str(worker_dir) not in sys.path:
    sys.path.insert(0, str(worker_dir))

importlib.invalidate_caches()
workload_worker = importlib.import_module("workload_worker")


def direct_process_cpu_once(pool):
    return list(pool.map(workload_worker.count_primes,
                         [PRIME_LIMIT] * CPU_TASKS,
                         chunksize=1))


async def awaited_process_cpu_once(pool):
    loop = asyncio.get_running_loop()
    futures = [
        loop.run_in_executor(pool, workload_worker.count_primes, PRIME_LIMIT)
        for _ in range(CPU_TASKS)
    ]
    return await asyncio.gather(*futures)


multiprocessing.freeze_support()
process_pool = ProcessPoolExecutor(max_workers=PROCESS_WORKERS)

try:
    # Start every worker before measuring.
    warmup = [process_pool.submit(workload_worker.count_primes, 2_000)
              for _ in range(PROCESS_WORKERS)]
    [f.result() for f in warmup]

    process_record = timed(
        f"persistent processes ({PROCESS_WORKERS} workers)",
        lambda: direct_process_cpu_once(process_pool),
    )

    # Futures are awaitable; a plain count_primes() result is not.
    awaited_process_result = await awaited_process_cpu_once(process_pool)
finally:
    process_pool.shutdown(wait=True)

benchmark_results["process"] = [process_record]
assert tuple(process_record["value"]) == tuple(serial_record["value"])
assert tuple(awaited_process_result) == tuple(serial_record["value"])

print_results("Process-pool CPU workload", [process_record])
print(describe_change(serial_record, process_record))
print("Correct await confirmed: the event loop awaited process-pool futures successfully.")

if serial_record["seconds_median"] / process_record["seconds_median"] <= 1.05:
    print("No speedup here: the visible workers gave no extra usable throughput, or overhead dominated.")


Process-pool CPU workload
-------------------------
persistent processes (1 workers)  median=3.3712s  samples=[3.4135, 3.7570, 3.3368, 3.0453, 3.3712]
persistent processes (1 workers): 1.07x slower than baseline
Correct await confirmed: the event loop awaited process-pool futures successfully.
No speedup here: the visible workers gave no extra usable throughput, or overhead dominated.


## The right way to `await` CPU work

Wrong:

```python
await count_primes(16_000)   # work finishes first, then Python tries to await an int
```

Right:

```python
future = loop.run_in_executor(process_pool, count_primes, 16_000)
result = await future
```

A worker process does the computing. The event loop only awaits a future that stands for "done", so the loop thread never runs the CPU function itself.

## Speedup against each baseline

Ratios above 1.00x mean faster than the baseline; below 1.00x means slower. Baselines: threads for the waiting tests, serial for the CPU tests.

In [9]:
# relative results, computed from benchmark_results.
comparisons = [
    ("1A equal waits",  fair_thread,   fair_async),
    ("1B 5,000 waits",  scale_thread,  scale_async),
    ("2 CPU, threads",  serial_record, thread_record),
    ("2 CPU, asyncio",  serial_record, async_record),
    ("3 CPU, processes", serial_record, process_record),
]

print(f"{'comparison':<18} {'baseline':>10} {'candidate':>10} {'speedup':>9}")
print("-" * 50)
for name, base, cand in comparisons:
    ratio = base["seconds_median"] / cand["seconds_median"]
    print(f"{name:<18} {base['seconds_median']:>9.4f}s {cand['seconds_median']:>9.4f}s {ratio:>8.2f}x")

comparison           baseline  candidate   speedup
--------------------------------------------------
1A equal waits        0.0619s    0.0545s     1.13x
1B 5,000 waits        1.0602s    0.1348s     7.87x
2 CPU, threads        3.1641s    3.1305s     1.01x
2 CPU, asyncio        3.1641s    3.0448s     1.04x
3 CPU, processes      3.1641s    3.3712s     0.94x


## All medians in one table

Read each group on its own:

- `waiting_equal`: fair overhead check
- `waiting_scale`: different capacity limits on purpose
- `cpu`: threads and asyncio should sit on the serial line
- `process`: speedup depends on usable CPU capacity

In [10]:
summary_rows = [
    (group, rec["label"], rec["seconds_median"])
    for group, records in benchmark_results.items()
    for rec in records
]

group_w = max(len(r[0]) for r in summary_rows)
model_w = max(len(r[1]) for r in summary_rows)

print(f"{'experiment':<{group_w}}  {'model':<{model_w}}  median seconds")
print(f"{'-' * group_w}  {'-' * model_w}  --------------")
for group, model, seconds in summary_rows:
    print(f"{group:<{group_w}}  {model:<{model_w}}  {seconds:14.4f}")

experiment     model                                               median seconds
-------------  --------------------------------------------------  --------------
waiting_equal  threads (256 tasks / 256 workers)                           0.0619
waiting_equal  asyncio (256 tasks / one loop thread)                       0.0545
waiting_scale  bounded thread pool (5000 jobs / 256 workers)               1.0602
waiting_scale  asyncio (5000 outstanding tasks / one loop thread)          0.1348
cpu            serial                                                      3.1641
cpu            threads (5 workers)                                         3.1305
cpu            asyncio gather (CPU tasks do not yield)                     3.0448
process        persistent processes (1 workers)                            3.3712


## Conclusions

- **Waiting work:** At equal concurrency the two models cost about the same. At 5,000 tasks the difference is capacity: a 256-thread pool works in waves, while one event loop holds every waiter at once.
- **Pure-Python CPU work:** Threads and asyncio stay on the serial line, because neither adds a core.
- **Process workers:** A persistent process pool can use multiple cores, as long as the runtime really provides them.

In short: **the event loop holds the waiters; worker processes spend the cores.**

### Before you trust this in production

- **Libraries differ.** Image and numeric libraries may release the GIL or use native threads. Benchmark the real library.
- **`os.cpu_count()` is a hint.** It reports visible logical CPUs, not guaranteed throughput.
- **Set limits.** Bound worker counts, queues, memory, retries and upload sizes.
- **Queues hide overload.** An unbounded queue postpones failure; it does not prevent it.
- **Re-measure after every design change.** The bottleneck moves.